# LangChain & LangGraph
Key concepts from the LangChain ecosystem demonstrated with pure Python.
Prerequisites: `pip install numpy matplotlib`

*Note: We simulate LangChain patterns without requiring the library itself.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Any, Callable

## 1. Chain Pattern
LangChain's core idea: **chain** together steps (prompts, LLM calls, tools).

In [ ]:
class SimpleChain:
    """Simulates a LangChain-style chain of operations."""
    def __init__(self, steps: List[Callable]):
        self.steps = steps
    
    def invoke(self, input_data):
        result = input_data
        for i, step in enumerate(self.steps):
            result = step(result)
            print(f'  Step {i+1}: {type(result).__name__} — {str(result)[:60]}')
        return result

# Build a chain: format prompt → simulate LLM → parse output
chain = SimpleChain([
    lambda x: f'Summarize this: {x}',            # Prompt template
    lambda x: f'Summary of input ({len(x)} chars)',  # Simulated LLM
    lambda x: {'output': x, 'length': len(x)},   # Output parser
])

print('Running chain:')
result = chain.invoke('Machine learning is a subset of AI...')
print(f'\nFinal result: {result}')

## 2. Prompt Templates
Structured prompts with variables that get filled at runtime.

In [ ]:
class PromptTemplate:
    """Simple prompt template with variable substitution."""
    def __init__(self, template: str, variables: List[str]):
        self.template = template
        self.variables = variables
    
    def format(self, **kwargs):
        missing = set(self.variables) - set(kwargs.keys())
        if missing:
            raise ValueError(f'Missing variables: {missing}')
        return self.template.format(**kwargs)

# Few-shot prompt
few_shot = PromptTemplate(
    template=(
        'Classify the sentiment of the review.\n\n'
        'Examples:\n'
        'Review: "Great product!" → Positive\n'
        'Review: "Terrible quality" → Negative\n\n'
        'Review: "{review}" → '
    ),
    variables=['review']
)

print(few_shot.format(review='The battery life is amazing'))

## 3. Retrieval Chain (RAG with LangChain Pattern)

In [ ]:
class RetrievalChain:
    """Simulates a retrieval-augmented generation chain."""
    def __init__(self, documents: List[str]):
        self.docs = documents
    
    def retrieve(self, query, top_k=2):
        # Simple keyword matching (in practice, use embeddings)
        scores = []
        for doc in self.docs:
            score = sum(1 for w in query.lower().split() if w in doc.lower())
            scores.append(score)
        indices = np.argsort(scores)[-top_k:][::-1]
        return [self.docs[i] for i in indices]
    
    def invoke(self, query):
        retrieved = self.retrieve(query)
        context = '\n'.join(retrieved)
        prompt = f'Context: {context}\n\nQuestion: {query}\nAnswer:'
        # Simulated LLM response
        return {'query': query, 'context': retrieved, 'answer': f'Based on context: {retrieved[0][:50]}...'}

docs = [
    'LangChain is a framework for building LLM applications.',
    'LangGraph adds stateful graph-based workflows to LangChain.',
    'Agents can use tools like search, calculators, and databases.',
    'RAG retrieves relevant documents before generating answers.',
]
rag = RetrievalChain(docs)
result = rag.invoke('What is LangGraph?')
print(f"Query: {result['query']}")
print(f"Retrieved: {result['context']}")
print(f"Answer: {result['answer']}")

## 4. LangGraph: State Machines for Agents
LangGraph models agent workflows as **directed graphs** with state.

In [ ]:
class StateGraph:
    """Simplified LangGraph-style state machine."""
    def __init__(self):
        self.nodes = {}      # name → function
        self.edges = {}      # name → list of (condition, target)
        self.entry = None
    
    def add_node(self, name, func):
        self.nodes[name] = func
    
    def add_edge(self, source, target, condition=None):
        self.edges.setdefault(source, []).append((condition, target))
    
    def set_entry(self, name):
        self.entry = name
    
    def run(self, state: Dict, max_steps=10):
        current = self.entry
        path = []
        
        for step in range(max_steps):
            if current is None or current == 'END':
                break
            path.append(current)
            state = self.nodes[current](state)
            
            # Find next node
            next_node = None
            for condition, target in self.edges.get(current, []):
                if condition is None or condition(state):
                    next_node = target
                    break
            current = next_node
        
        return state, path

# Build a research agent graph
graph = StateGraph()
graph.add_node('classify', lambda s: {**s, 'type': 'factual' if '?' in s['query'] else 'creative'})
graph.add_node('search', lambda s: {**s, 'results': ['Result A', 'Result B']})
graph.add_node('generate', lambda s: {**s, 'answer': f"Answer based on {len(s.get('results',[]))} sources"})
graph.add_node('create', lambda s: {**s, 'answer': 'Creative response generated'})

graph.set_entry('classify')
graph.add_edge('classify', 'search', condition=lambda s: s['type'] == 'factual')
graph.add_edge('classify', 'create', condition=lambda s: s['type'] == 'creative')
graph.add_edge('search', 'generate')
graph.add_edge('generate', 'END')
graph.add_edge('create', 'END')

state, path = graph.run({'query': 'What is machine learning?'})
print(f'Path: {" → ".join(path)}')
print(f'Final state: {state}')

## 5. Visualizing the Graph

In [ ]:
# Visualize the agent graph
fig, ax = plt.subplots(figsize=(8, 4))
positions = {'classify': (1, 2), 'search': (2, 3), 'create': (2, 1),
             'generate': (3, 3), 'END': (4, 2)}
for name, (x, y) in positions.items():
    color = '#4CAF50' if name == 'END' else '#2196F3'
    ax.scatter(x, y, s=2000, c=color, zorder=5)
    ax.text(x, y, name, ha='center', va='center', fontweight='bold', fontsize=9, color='white')

edges = [('classify','search'), ('classify','create'), ('search','generate'), ('generate','END'), ('create','END')]
for src, tgt in edges:
    x1,y1 = positions[src]; x2,y2 = positions[tgt]
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_xlim(0.5, 4.5); ax.set_ylim(0.5, 3.5)
ax.set_title('LangGraph-Style Agent Workflow'); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Output Parsers

In [ ]:
import re

class JSONOutputParser:
    """Parse structured output from LLM text."""
    def parse(self, text):
        # Try to extract JSON from text
        match = re.search(r'\{[^}]+\}', text)
        if match:
            import json
            return json.loads(match.group())
        return {'raw': text}

parser = JSONOutputParser()
# Simulated LLM output
llm_output = 'The sentiment is: {"label": "positive", "confidence": 0.95}'
parsed = parser.parse(llm_output)
print(f'Raw: {llm_output}')
print(f'Parsed: {parsed}')

## 7. Interview Takeaways
- **LangChain** chains together prompts, LLMs, tools, and parsers
- **LCEL** (LangChain Expression Language) uses `|` pipe syntax for composability
- **LangGraph** adds stateful, graph-based workflows with conditional routing
- Key patterns: **Chains**, **Agents**, **RAG**, **Memory**
- LangGraph enables cycles (loops) for iterative agent behavior
- **LangSmith** provides observability (tracing, evaluation) for LLM apps

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>